# Scientific Reasoning Assistant: TinyLlama-1.1B + QLoRA (Colab, free T4)

Fine-tunes a LoRA adapter on a 4-bit TinyLlama-1.1B-Chat so that, given a passage from a
paper, it answers questions about claims, assumptions, limitations and follow-up
experiments, and can produce a JSON "reasoning pack". Only the adapter is saved.

Runtime: Runtime > Change runtime type > T4 GPU. Training data (`data/examples.json`)
must be provided; see `data/README.md`.

In [ ]:
# 1. Get the code and install dependencies (keeps Colab's preinstalled torch)
!git clone -q https://github.com/bioinfosourabh/qlora-scientific-reasoning.git
%cd qlora-scientific-reasoning
!grep -v '^torch==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt && pip install -q -e .

In [ ]:
# 2. GPU check
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Training data
Upload `examples.json` (format in `data/README.md`), or copy it from Google Drive.

In [ ]:
# 3. Provide and validate the examples
import os, shutil
if not os.path.exists("data/examples.json"):
    from google.colab import files
    up = files.upload()                      # choose examples.json
    shutil.move(next(iter(up)), "data/examples.json")
from sciassist.data import load_examples, validate, split
ex = load_examples("data/examples.json")
problems = validate(ex)
print(len(ex), "examples;", "OK" if not problems else problems)
train, val = split(ex, 0.15)
print(len(train), "train /", len(val), "validation")

In [ ]:
# 4. QLoRA training (4-bit NF4, double quantisation, fp16 compute; LoRA r=16, alpha=32 on q/k/v/o)
!python -m sciassist.train --config configs/train_tinyllama_qlora.yaml

In [ ]:
# 5. Evaluate on the held-out examples (greedy decoding)
!python -m sciassist.evaluate --adapter outputs/sci-reasoning-assistant/adapter \
    --examples data/examples.json --split outputs/sci-reasoning-assistant/split.json --out outputs/eval

In [ ]:
# 6. Interactive session: paste text or upload a PDF, then ask questions
from sciassist.assistant import ScientificReasoningAssistant, load_model
model, tok = load_model("outputs/sci-reasoning-assistant/adapter")
assistant = ScientificReasoningAssistant(model, tok)

TEXT = """Paste an abstract or section here."""
paper = assistant.load_paper(text=TEXT)
# or: from google.colab import files; pdf = next(iter(files.upload())); paper = assistant.load_paper(pdf_path=pdf)
print(paper.title, len(paper.full_text), "characters")

In [ ]:
print(assistant.chat("What are the main limitations of this study?"))

In [ ]:
print(assistant.chat("What experiments would strengthen these findings?"))

In [ ]:
# Structured mode: JSON reasoning pack, parsed with repair and validated
from sciassist.structured import parse_pack, grounding
pack, parsed, errors = parse_pack(assistant.reasoning_pack())
print(pack.model_dump_json(indent=2) if pack else parsed.obj, errors)

In [ ]:
# 7. Save the session and the adapter
open("session.md", "w").write(assistant.export_markdown())
!zip -qr adapter.zip outputs/sci-reasoning-assistant/adapter
from google.colab import files
files.download("adapter.zip"); files.download("session.md")

In [ ]:
# Optional: Gradio interface with a public link (stop the cell to end it)
!python -m sciassist.app --adapter outputs/sci-reasoning-assistant/adapter --share